# Paired protocol: CIFAR-100 (Tables 1 and 2)

DA351 Phase 1, FVLC replication (Lai et al., ICLR 2024). This is the notebook we actually ran on **Kaggle** (GPU T4), with its outputs.

The repo was attached as a Kaggle dataset (Add Input). The results zip it produced is `results_paired/cifar100/` in this repo.
Only Kaggle's default boilerplate cell was removed, and pip progress bars were trimmed from the output; nothing else was changed.


In [2]:
import os, glob, shutil
os.makedirs('/tmp/code', exist_ok=True)
z = glob.glob('/kaggle/input/**/*fvlc-replication*.zip', recursive=True)
if z:
    !unzip -q -o "{z[0]}" -d /tmp/code/
else:
    src = os.path.dirname(glob.glob('/kaggle/input/**/run_experiments.py', recursive=True)[0])
    shutil.copytree(src, '/tmp/code/fvlc-replication', dirs_exist_ok=True)
%cd /tmp/code/fvlc-replication

!pip install -q ftfy regex tqdm nltk "pytorchcv>=0.0.67"
!pip install -q git+https://github.com/openai/CLIP.git
!python -c "import clip, pytorchcv, nltk; print('PACKAGES OK')"

# CIFAR: reuse a Kaggle copy if one is attached (avoids the slow 40-min download)
os.environ['FVLC_DATA_ROOT'] = '/tmp/data'
os.makedirs('/tmp/data', exist_ok=True)
for name in ['cifar-10-batches-py', 'cifar-100-python']:
    hit = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hit and not os.path.exists(f'/tmp/data/{name}'):
        shutil.copytree(hit[0], f'/tmp/data/{name}')
        print('Using Kaggle copy of', name)

os.environ['FVLC_ACT_DIR'] = '/tmp/activations'
OUT = '/kaggle/working/results_paired'
!grep -c "def evaluate_paired" fvlc/pipeline.py
!python tests/test_core.py

/tmp/code/fvlc-replication
  Preparing metadata (setup.py) ... done
PACKAGES OK
1
100%|██████████████████████████████████████████| 50/50 [00:00<00:00, 103.23it/s]
did not converge at 50 iterations (criteria False)
(0) lambda 0.0007, loss 0.0270, acc 1.0000 [val acc 1.0000] [test acc -1.0000], sparsity 0.9791666666666666 [47/48], time 0.49999213218688965, lr 0.1000
universal: TCPC base 1.338e-01 -> FVLC 9.998e-03 | TOPC 5.930e-02 -> 4.852e-03
per_sample: TCPC base 1.338e-01 -> FVLC 1.890e-02 | TOPC 5.930e-02 -> 8.786e-03
all tests passed


In [3]:
!python run_experiments.py --dataset cifar100 --protocol paired --fvlc_init gaussian --out $OUT
!python make_tables.py --results $OUT

Dataset cifar100 | device cuda | seed 42
100%|█████████████████████████████████████████| 169M/169M [09:11<00:00, 306kB/s]
100%|███████████████████████████████████████| 244M/244M [00:02<00:00, 96.2MiB/s]
100%|████████████████████████████████████████| 335M/335M [00:01<00:00, 186MiB/s]
[vanilla] 892 concepts -> 857 after CLIP cutoff                                 
  projection: best step 200, val similarity 0.6005
[vanilla] 827 concepts after interpretability cutoff
 21%|████████▎                               | 209/1000 [03:03<11:34,  1.14it/s]
(0) lambda 0.0007, loss 1.5077, acc 0.6780 [val acc 0.6509] [test acc -1.0000], sparsity 0.034740024183796854 [2873/82700], time 184.154944896698, lr 0.1000
[vanilla] test acc 0.6509  (2873/82700 non-zero final weights)
  6%|██▍                                      | 60/1000 [00:58<15:14,  1.03it/s]
(0) lambda 0.0030, loss 2.8955, acc 0.5995 [val acc 0.5845] [test acc -1.0000], sparsity 0.031142578125 [3189/102400], time 58.9543559551239, lr 0.10

In [4]:
!cd /kaggle/working && zip -qr results_cifar100.zip results_paired -x "*/layers/*" "*/vanilla.pt"
print(open(f'{OUT}/tables.md').read())

# Replication results vs. paper

Ours = this repo (seed 42, evaluation protocol: paired). Paper = Lai et al., ICLR 2024. Accuracy in %.


## Headline numbers (averaged over the 7 paper perturbations)

| Dataset | LF-CBM acc ours / paper | FVLC acc ours / paper | base TCPC ours / paper | FVLC TCPC ours / paper | base TOPC ours / paper | FVLC TOPC ours / paper | seed-only TCPC ours |
|---|---|---|---|---|---|---|---|
| CIFAR-100 | 65.09 / 65.42 | 64.96 / 65.21 | 3.50e-01 / 1.59e-01 | 2.18e-02 / 8.75e-03 | 2.08e-01 / 1.02e-01 | 7.26e-03 / 7.83e-03 | 2.39e-01 |

## Table 1 – Accuracy

| Method | CIFAR-100 ours | CIFAR-100 paper |
|---|---|---|
| Standard (no interpretability) | 58.45 | 70.10 |
| Label-free CBM | 65.09 | 65.42 |
| WP1(5%) – base | 61.65 | 65.13 |
| WP1(5%) – FVLC | 64.91 | 65.43 |
| WP1(10%) – base | 60.58 | 65.09 |
| WP1(10%) – FVLC | 64.91 | 64.90 |
| WP2 – base | 64.52 | 65.16 |
| WP2 – FVLC | 64.91 | 65.34 |
| IP – base | 64.50 | 65.36 |
| IP – FVLC | 65.07 | 65.29 |
| 